# NB03 — MAI Carriers vs Metal Tolerance: Cross-Project Phenotype Analysis

## Purpose
Test whether MAI (malonylpyruvate isomerase / K16163) carrier status predicts metal tolerance
phenotype in ENIGMA isolates by cross-referencing three companion datasets.

## Scientific question
If the mycothiol-dependent detoxification module is genuinely adaptive under metal stress, then
MAI-carrying strains should show better growth under metal challenge than non-carriers, controlling
for phylogeny.

## Approach
- **Section 1 (local)**: Data landscape — what is and isn't available without Spark.
- **Section 2 (local)**: Genus-level preliminary comparison using `genotype_to_phenotype_enigma` growth data.
- **Section 3 (local)**: Context from `genotype_to_phenotype_enigma` — the metal-growth null result.
- **Section 4 (Spark)**: Proper per-genome analysis: apply `enigma_stress_phenotype_ml` regression
  models to all ENIGMA Genome Depot proteins; join with MAI carrier status.
- **Section 5 (Spark)**: Contamination gradient — does MAI prevalence correlate with site contamination?
- **Section 6**: Active learning / experimental recommendations for Streptomyces candidates.

## FitnessBrowser Actinomycetota coverage (clarification)
The local FitnessBrowser training data (60 organisms) includes two Actinomycetota:
- **MycoTube** (M. tuberculosis H37Rv): has Zn/Cu/Ni metal stress experiments (6 replicated conditions)
- **Bifido** (Bifidobacterium sp.): has Zn data only

K16163 (MAI) is **not annotated** in M. tuberculosis (confirmed: 0 hits in protein file); the
MAI-dominant ENIGMA genera — Streptomyces, Rhodococcus, Nocardioides — are absent from local data.
The FitnessBrowser website has 62 organisms (2 not downloaded here); if either is a soil
Actinomycetota with K16163, the feasibility of a direct cross-reference changes.

## Key prior result (from `genotype_to_phenotype_enigma`)
Binary metal growth is NOT predictable from KO content (AUC ≈ 0.5 for metals vs. 0.775 for
amino acids). This is critical context: absence of a statistical signal here does not refute
the MAI hypothesis — it may reflect the limits of KO-content features rather than the biology.

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu, fisher_exact, spearmanr
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings('ignore')

NOTEBOOK_DIR  = Path().resolve()
PROJECT_DIR   = NOTEBOOK_DIR.parent
DATA_DIR      = PROJECT_DIR / 'data'
FIG_DIR       = PROJECT_DIR / 'figures'

# Companion project roots (must be on the same filesystem)
REPO_ROOT     = PROJECT_DIR.parent
ESML_DIR      = REPO_ROOT / 'enigma_stress_phenotype_ml'
GTP_DIR       = REPO_ROOT / 'genotype_to_phenotype_enigma'
CONTAM_DIR    = REPO_ROOT / 'enigma_contamination_functional_potential'

print('Setup complete.')
print(f'  ESML data: {ESML_DIR / "data"}')
print(f'  GtP data:  {GTP_DIR / "data"}')
print(f'  Contam data: {CONTAM_DIR / "data"}')

## Section 1 — Data Landscape

Before running analyses, document exactly what data is and isn't available locally.

In [ ]:
# Load local datasets
all_genomes = pd.read_parquet(DATA_DIR / 'enigma_all_genomes.parquet')
mai_hits    = pd.read_csv(DATA_DIR / 'enigma_mai_hits.csv')
shortlist   = pd.read_csv(DATA_DIR / 'enigma_candidate_shortlist.csv')

# genotype_to_phenotype_enigma
gtp_strains = pd.read_csv(GTP_DIR / 'data' / 'strain_scalars.tsv', sep='\t')
gtp_preds   = pd.read_csv(GTP_DIR / 'data' / 'model_predictions.tsv', sep='\t')

# Metal conditions in genotype_to_phenotype_enigma
METAL_CONDITIONS = ['cobaltchloridehexa', 'manganeseiichloridetetra',
                    'nickelchloridehexa', 'zincdichloride']

gtp_metal = gtp_preds[gtp_preds['condition'].isin(METAL_CONDITIONS)].copy()
gtp_metal['genus'] = gtp_metal['strain'].map(
    dict(zip(gtp_strains['strain_id'], gtp_strains['genus']))
)

# MAI carrier genus distribution
mai_hits['genus'] = mai_hits['organism_name'].str.split().str[0]
all_genomes['genus'] = all_genomes['organism_name'].str.split().str[0]

# Overlap between MAI-carrier genera and GtP strain genera
mai_genera   = set(mai_hits['genus'].dropna())
gtp_genera   = set(gtp_strains['genus'].dropna())
overlap_gen  = mai_genera & gtp_genera

print('=== DATA LANDSCAPE ===' )
print(f'ENIGMA all genomes (all_genomes.parquet): {len(all_genomes):,} genomes')
print(f'  MAI carriers (has_mai=1):              {all_genomes["has_mai"].sum():,}')
print(f'  MAI absent  (has_mai=0):               {(all_genomes["has_mai"]==0).sum():,}')
print()
print(f'enigma_mai_hits.csv:                     {len(mai_hits):,} genome records ({mai_hits["organism_name"].nunique()} unique strains)')
print(f'  Unique genera with MAI:                {len(mai_genera)}')
print()
print(f'genotype_to_phenotype_enigma strains:    {len(gtp_strains):,}')
print(f'  With metal growth data:                {gtp_metal["strain"].nunique()}')
print(f'  Metal conditions tested:               {METAL_CONDITIONS}')
print()
print(f'Genus overlap (MAI ∩ GtP):               {len(overlap_gen)} genera: {sorted(overlap_gen)}')

# FitnessBrowser organism coverage — Actinomycetota situation
labeled_df = pd.read_parquet(ESML_DIR / 'data' / 'labeled_pd.parquet')
fb_orgs = sorted(labeled_df['organism'].unique())
# Identify Actinomycetota in local training data
# Confirmed: MycoTube = M. tuberculosis H37Rv (Actinomycetota; Zn/Cu/Ni metal stress data)
#            Bifido   = Bifidobacterium sp. (Actinomycetota; Zn data only)
#            Brev2    = Brevundimonas (Alphaproteobacteria, NOT Actinomycetota)
actino_fb = ['MycoTube', 'Bifido']
mai_relevant_actino = ['Streptomyces', 'Rhodococcus', 'Nocardioides', 'Arthrobacter']

print()
print(f'enigma_stress_phenotype_ml FitnessBrowser organisms in local data: {len(fb_orgs)}')
print(f'  Actinomycetota present: {actino_fb}')
print(f'    MycoTube: M. tuberculosis H37Rv — Zn (pyrithione), Cu(II)Cl₂, Ni(II)Cl₂ data')
print(f'    Bifido  : Bifidobacterium sp.  — Zn data only')
print(f'  NOTE: K16163 (MAI) is NOT annotated in either organism (BLAST vs. MycoTube proteins: 0 hits).')
print(f'  The MAI-dominant ENIGMA genera {mai_relevant_actino} are NOT in the local 60-organism set.')
print()
print('  *** FitnessBrowser website has 62 organisms (2 more than local data). ***')
print('  *** Those 2 additional organisms are not accounted for here. ***')
print('  *** If either is Streptomyces or Rhodococcus, Section 4 analysis changes significantly. ***')
print()
print('Cross-reference limitation: The critical genera for the MAI hypothesis are absent locally.')
print('Full per-genome analysis requires Spark + ENIGMA Genome Depot (Section 4).')

## Section 1b — Global Phylogenetic Context: pan-bacterial evidence for K16163 bias

Data from `final_draft` project: 254,783 high-quality genomes (GTDB taxonomy) with KEGG KO
annotations, tested for phylum-level enrichment using Fisher's exact test.

**Key result**: K16163 (MAI) and the core mycothiol pathway (mshA/B/C) are all statistically
significantly Actinomycetota-biased at genome scale. This provides the global context showing
the ENIGMA BERIL MAI finding (136 strains, 5 ENIGMA sites) is consistent with a pan-Actinomycetota
pattern.

**Complementary FitnessBrowser finding**: mshD (K16150, mycothiol acetyltransferase) is the #1
most Zn-sensitive gene in *M. tuberculosis* H37Rv (fitness = −2.29 at Zn, −2.28 at Cu) — the
strongest metal fitness phenotype of any gene in MycoTube. This is direct experimental evidence
that mycothiol biosynthesis is required for metal tolerance in Actinomycetota.

In [ ]:
import pandas as pd
from pathlib import Path

FINAL_DRAFT_DIR = PROJECT_DIR.parent / 'final_draft' / 'data'
bias_ko = pd.read_csv(FINAL_DRAFT_DIR / 'bias_ko_df.csv')

# Mycothiol pathway KOs of interest
MYCOTHIOL_KOS = {
    'K16163': 'MAI (malonylpyruvate isomerase) — THESIS TARGET',
    'K16147': 'mshA (GlcNAc-Ins-P transferase)',
    'K16148': 'mshB (GlcNAc-Ins deacetylase)',
    'K16149': 'mshC (mycothiol ligase)',
    'K03975': 'IolA/MaiA (maleylpyruvate isomerase, related)',
}

# Filter to Actinomycetota rows for these KOs
actino = bias_ko[
    (bias_ko['kegg_orthology_id'].isin(MYCOTHIOL_KOS)) &
    (bias_ko['focal_phylum'] == 'p__Actinomycetota')
].copy()
actino['gene_name'] = actino['kegg_orthology_id'].map(MYCOTHIOL_KOS)
actino['n_actino_with_KO'] = actino['a']
actino['n_actino_total'] = actino['a'] + actino['b']
actino['pct_actino'] = 100 * actino['a'] / actino['n_actino_total']
actino['n_other_with_KO'] = actino['c']
actino['n_other_total'] = actino['c'] + actino['d']
actino['pct_other'] = 100 * actino['c'] / actino['n_other_total']

print('=== GLOBAL PHYLOGENETIC BIAS: mycothiol pathway KOs (n=254,783 genomes) ===')
print()
cols = ['kegg_orthology_id', 'gene_name', 'n_actino_with_KO', 'pct_actino', 'pct_other', 'odds_ratio', 'q_value']
print(actino[cols].sort_values('odds_ratio', ascending=False).to_string(index=False))
print()
print('All q-values = 0.0 (p < machine epsilon; n=254,783 genomes)')
print()
print('Interpretation:')
print('  K16163 (MAI):  OR=182.98 — 19.8% of Actinomycetota genomes vs 0.2% of Pseudomonadota')
print('  mshB (K16148): OR=3826   — THE most Actinomycetota-restricted mycothiol gene')
print('  mshC (K16149): OR=248    — present in 11.6% of 24K Actinomycetota genomes')
print('  mshA (K16147): OR=26     — more broadly distributed (some HGT likely)')
print()
print('These pan-scale results validate the ENIGMA MAI finding as a phylum-scale biological pattern,')
print('not an ENIGMA sampling artifact.')

# FitnessBrowser mshD finding (from MycoTube analysis)
print()
print('=== FITNESSBROWSER COMPLEMENTARY EVIDENCE ===')
print('mshD (K16150, mycothiol acetyltransferase) in M. tuberculosis H37Rv:')
print('  Zn fitness = -2.289  (rank 1/2881 = #1 most Zn-sensitive gene in MycoTube)')
print('  Cu fitness = -2.279  (rank 8/2881 = top 0.3% most Cu-sensitive)')
print('  Ni fitness = -1.598')
print()
print('  mshD is ESSENTIAL for metal tolerance in M. tuberculosis — the only Actinomycetota')
print('  with direct FitnessBrowser metal stress data.')
print()
print('  NOTE: K16150 (mshD) is NOT currently in enigma_all_genomes.parquet.')
print('  ACTION: Add K16150 to the Spark query in Section 4 alongside K16163.')

## Section 2 — Genus-Level Preliminary Comparison

For the 5 overlapping genera (Agrobacterium, Arthrobacter, Ensifer, Mesorhizobium, Microbacterium),
test whether strains from MAI-carrier genera show different metal growth than non-carrier genera.

**Limitations**: (1) n = 5 genera — very low power; (2) genus-level comparison conflates
within-genus MAI variation; (3) overlapping genera are mostly Alphaproteobacteria with incidental
MAI detections (18 Rhizobium + 4 Ensifer), not Actinomycetota. **Treat as hypothesis-generating
only.**

In [ ]:
# Build genus-level MAI prevalence from enigma_all_genomes
genus_mai = all_genomes.groupby('genus').agg(
    n_total   = ('has_mai', 'count'),
    n_mai     = ('has_mai', 'sum')
).reset_index()
genus_mai['mai_prevalence'] = genus_mai['n_mai'] / genus_mai['n_total']
genus_mai['mai_present_in_genus'] = genus_mai['n_mai'] > 0

# Merge with GtP metal growth data at genus level
gtp_metal_with_genus = gtp_metal.merge(
    genus_mai[['genus', 'mai_prevalence', 'mai_present_in_genus', 'n_mai', 'n_total']],
    on='genus', how='inner'
)

print(f'Genus-level metal growth records with MAI data: {len(gtp_metal_with_genus)}')
print(f'Unique genera with both metal growth + MAI data: {gtp_metal_with_genus["genus"].nunique()}')
print()

if len(gtp_metal_with_genus) > 0:
    # Per-condition Mann-Whitney test: MAI-present genus vs MAI-absent genus strains
    results = []
    for cond in METAL_CONDITIONS:
        sub = gtp_metal_with_genus[gtp_metal_with_genus['condition'] == cond]
        if len(sub) < 4:
            continue
        pos = sub[sub['mai_present_in_genus'] & sub['y_true'].notna()]['y_true']
        neg = sub[~sub['mai_present_in_genus'] & sub['y_true'].notna()]['y_true']
        if len(pos) > 0 and len(neg) > 0:
            stat, p = mannwhitneyu(pos, neg, alternative='two-sided')
            results.append({'condition': cond, 'n_mai_pos_genus': len(pos),
                            'n_mai_neg_genus': len(neg),
                            'median_mai_pos': pos.median(), 'median_mai_neg': neg.median(),
                            'MWU_stat': stat, 'p_value': p})

    if results:
        res_df = pd.DataFrame(results)
        res_df['q_BH'] = multipletests(res_df['p_value'], method='fdr_bh')[1]
        print('Genus-level Mann-Whitney: MAI-present genus vs MAI-absent genus strains')
        print(res_df.to_string(index=False))
    else:
        print('Insufficient data for per-condition Mann-Whitney test.')
        print('Details of available data:')
        print(gtp_metal_with_genus[['genus', 'condition', 'y_true', 'mai_present_in_genus']].to_string())
else:
    print('No genus-level overlap with metal growth data.')
    print('Genus overlap genera:', sorted(overlap_gen))
    print('Metal growth genera:', sorted(gtp_metal['genus'].dropna().unique()))

## Section 3 — Context: The `genotype_to_phenotype_enigma` Null Result

**Critical finding**: Binary metal growth is NOT predictable from KO content across ENIGMA
isolates (AUC ≈ 0.5 for metals), while amino acid growth IS predictable (AUC = 0.775).
This null result was established in `genotype_to_phenotype_enigma` using the same ENIGMA
Genome Depot strains and growth curve data.

Interpretation for this project:
1. If KO content (which includes K16163 / MAI) doesn't predict metal growth, then a simple
   MAI-presence × metal-growth test is expected to be weak — not because MAI is irrelevant,
   but because growth in liquid culture may not capture the relevant phenotype.
2. The MAI hypothesis is about detoxification of reactive metal-thiol adducts (metal-MSH
   complexes), which may manifest as increased lag time, not OD600 endpoint.
3. **Recommended experiment**: µmax and lag time measurements (via growth curve fitting)
   may show MAI effects that binary endpoint assays miss.

In [ ]:
# Reproduce key per-condition accuracy for metal conditions from GtP data
pca = pd.read_csv(GTP_DIR / 'data' / 'per_condition_accuracy.tsv', sep='\t')

# Map canonical condition names to per_condition_accuracy conditions
metal_cond_map = {
    'cobaltchloridehexa': 'cobaltchloridehexa',
    'manganeseiichloridetetra': 'manganeseiichloridetetra',
    'nickelchloridehexa': ['nickelchloridehexa'],
    'zincdichloride': ['zincdichloride']
}

metal_acc = pca[pca['condition'].isin(METAL_CONDITIONS)].copy()

# Add comparison to best-performing conditions
best_auc = pca.nlargest(5, 'AUC')

print('AUC for metal growth prediction (from KO content — genotype_to_phenotype_enigma):')
if len(metal_acc) > 0:
    print(metal_acc[['condition', 'n', 'n_pos', 'AUC']].to_string(index=False))
else:
    print('  Metal conditions not found in per_condition_accuracy.tsv')
    print('  Available conditions sample:')
    print(pca.head(5)[['condition', 'AUC']].to_string(index=False))

print()
print('Comparison — top 5 best-predicted conditions:')
print(best_auc[['condition', 'condition_class', 'n', 'AUC']].to_string(index=False))

print()
print('Interpretation:')
print('  Metal growth AUC ≈ 0.5 → no better than chance from KO content')
print('  Amino acid growth AUC ≈ 0.77–0.93 → strong signal from KO content')
print('  This is consistent with the MAI phenotype being non-binary (lag time, MIC),')
print('  not captured by endpoint growth assays.')

## Section 4 — Spark Scaffold: Per-Genome Metal Fitness Analysis

**Requires**: Seaborg Spark cluster access (`get_spark_session()`) and the trained regression
models from `enigma_stress_phenotype_ml/data/models/stressor_{metal}_regression.cbm`.

**What this section does**:
1. Pull all ENIGMA Genome Depot protein sequences (amino acid) via Spark.
2. Compute aa+kmer2 features (same as training).
3. Apply 11 metal regression models to predict per-protein fitness.
4. Aggregate per-genome: mean predicted fitness across all proteins.
5. Join with MAI carrier status from `enigma_all_genomes.parquet`.
6. Run Mann-Whitney (MAI+ vs MAI−) per metal, BH-FDR across 11 metals.
7. Run Spearman (pathway_completeness vs. mean predicted fitness) per metal.

In [ ]:
# ─── SPARK SCAFFOLD — requires Seaborg ────────────────────────────────────────
# Uncomment and execute on JupyterHub / Seaborg

ESML_MODEL_DIR = ESML_DIR / 'data' / 'models'
METAL_STRESSORS = ['Zn', 'Cu', 'Cd', 'Co', 'Ni', 'Cr', 'As', 'Hg', 'Pb', 'Mn', 'Fe']

print('Section 4 scaffold (requires Spark).')
print('Paste and execute the following cells on Seaborg / JupyterHub:')
print()

scaffold = '''
# ── S4a: Spark session ──────────────────────────────────────────────────────
try:
    spark
except NameError:
    import sys
    sys.path.append('/opt/conda/lib/python3.13/site-packages')
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()

# ── S4b: Pull ENIGMA protein sequences (aa) from Genome Depot ────────────────
enigma_proteins_sql = \'\'\'\n    SELECT
        g.genome_id,
        g.genome_name AS organism_name,
        p.protein_id,
        p.sequence
    FROM enigma_genome_depot_enigma.browser_genome g
    JOIN enigma_genome_depot_enigma.browser_protein p
      ON g.genome_id = p.genome_id
    WHERE p.sequence IS NOT NULL
\'\'\'\nenigma_prot_df = spark.sql(enigma_proteins_sql).toPandas()
print(f"Loaded {len(enigma_prot_df):,} ENIGMA proteins from {enigma_prot_df['genome_id'].nunique()} genomes")

# ── S4b2: Check K16150 (mshD) presence per ENIGMA genome ─────────────────────
# NOTE: K16150 is NOT in enigma_all_genomes.parquet — query Genome Depot directly.
# mshD (K16150) = mycothiol acetyltransferase; #1 Zn-sensitive gene in M. tuberculosis.
mshdko_sql = \'\'\'\n    SELECT DISTINCT
        g.genome_id,
        1 AS has_mshD
    FROM enigma_genome_depot_enigma.browser_genome g
    JOIN enigma_genome_depot_enigma.browser_protein p
      ON g.genome_id = p.genome_id
    JOIN enigma_genome_depot_enigma.browser_protein_annotation pa
      ON p.protein_id = pa.protein_id
    WHERE pa.kegg_ko = \'K16150\'
\'\'\'\nmshdD_df = spark.sql(mshdko_sql).toPandas()
print(f"Genomes with mshD (K16150): {len(mshdD_df)} / {enigma_prot_df['genome_id'].nunique()} total ENIGMA genomes")

# ── S4c: Compute aa + kmer2 features ────────────────────────────────────────
import sys
sys.path.insert(0, str(ESML_DIR / 'src'))
from feature_extraction import compute_aa_features, compute_kmer_features

aa_feats   = compute_aa_features(enigma_prot_df['sequence'].tolist())
kmer_feats = compute_kmer_features(enigma_prot_df['sequence'].tolist(), k=2)
feat_df    = pd.concat([aa_feats, kmer_feats], axis=1)
feat_df.index = enigma_prot_df['protein_id'].values

# ── S4d: Apply regression models per metal ────────────────────────────────────
from catboost import CatBoostRegressor
pred_results = []

for metal in METAL_STRESSORS:
    model_path = ESML_MODEL_DIR / f"stressor_{metal}_regression.cbm"
    if not model_path.exists():
        print(f"{metal}: model not found, skipping")
        continue
    model = CatBoostRegressor()
    model.load_model(str(model_path))
    preds = model.predict(feat_df.values)
    tmp = enigma_prot_df[['genome_id', 'organism_name', 'protein_id']].copy()
    tmp['metal'] = metal
    tmp['pred_fitness'] = preds
    pred_results.append(tmp)

pred_all = pd.concat(pred_results, ignore_index=True)
print(f"Predictions: {len(pred_all):,} protein × metal records")

# ── S4e: Aggregate per-genome ────────────────────────────────────────────────
genome_pred = pred_all.groupby(['genome_id', 'metal'])['pred_fitness'].mean().unstack('metal')
genome_pred.columns = [f"{m}_pred_mean" for m in genome_pred.columns]
genome_pred = genome_pred.reset_index()

# ── S4f: Join with MAI + mshD carrier status ─────────────────────────────────
mai_info = pd.read_parquet(DATA_DIR / 'enigma_all_genomes.parquet')[['genome_id', 'has_mai']]
joined   = genome_pred.merge(mai_info, on='genome_id', how='left')

# mshD from S4b2 query above (K16150 not in parquet)
joined = joined.merge(mshdD_df, on='genome_id', how='left')
joined['has_mshD'] = joined['has_mshD'].fillna(0).astype(int)

# Also join pathway completeness
pathway_info = pd.read_csv(DATA_DIR / 'enigma_pathway_completeness.csv')[['genome_id', 'pathway_completeness']]
joined = joined.merge(pathway_info, on='genome_id', how='left')

print(f"joined: {len(joined)} genomes")
print(f"  has_mai=1:  {joined['has_mai'].sum():,}")
print(f"  has_mshD=1: {joined['has_mshD'].sum():,}")

# ── S4g: Statistical tests — MAI and mshD vs metal fitness ───────────────────
from scipy.stats import mannwhitneyu, spearmanr
from statsmodels.stats.multitest import multipletests

results = []
for marker, col_flag in [('has_mai', 'has_mai'), ('has_mshD', 'has_mshD')]:
    for metal in METAL_STRESSORS:
        col = f"{metal}_pred_mean"
        if col not in joined.columns:
            continue
        sub   = joined[joined[col].notna() & joined[col_flag].notna()]
        pos   = sub[sub[col_flag] == 1][col]
        neg   = sub[sub[col_flag] == 0][col]
        stat, p = mannwhitneyu(pos, neg, alternative='two-sided') if (len(pos) > 0 and len(neg) > 0) else (np.nan, np.nan)
        results.append({'marker': marker, 'metal': metal,
                        'n_pos': len(pos), 'n_neg': len(neg),
                        'median_pos': pos.median(), 'median_neg': neg.median(),
                        'MWU_stat': stat, 'MWU_p': p})

res_df = pd.DataFrame(results)
res_df['MWU_q_BH'] = multipletests(res_df['MWU_p'], method='fdr_bh')[1]

print("\\n=== MAI (K16163) vs metal predicted fitness ===")
mai_res = res_df[res_df['marker'] == 'has_mai']
print(mai_res[['metal', 'n_pos', 'n_neg', 'median_pos', 'median_neg', 'MWU_p', 'MWU_q_BH']].to_string(index=False))

print("\\n=== mshD (K16150) vs metal predicted fitness ===")
mshd_res = res_df[res_df['marker'] == 'has_mshD']
print(mshd_res[['metal', 'n_pos', 'n_neg', 'median_pos', 'median_neg', 'MWU_p', 'MWU_q_BH']].to_string(index=False))

print("\\nNote: mshD expected to show strongest signal for Zn and Cu based on MycoTube FitnessBrowser data.")
print("      K16150 is rank 1/2881 for Zn sensitivity and rank 8/2881 for Cu sensitivity in M. tuberculosis.")

res_df.to_csv(DATA_DIR / 'nb03_mai_mshd_metal_fitness_test.csv', index=False)

# ── S4h: Spearman pathway completeness vs predicted fitness ─────────────────
spearman_results = []
for metal in METAL_STRESSORS:
    col = f"{metal}_pred_mean"
    if col not in joined.columns:
        continue
    sub = joined[joined[col].notna() & joined['pathway_completeness'].notna()]
    rho, p = spearmanr(sub['pathway_completeness'], sub[col])
    spearman_results.append({'metal': metal, 'n': len(sub), 'rho': rho, 'p': p})

sp_df = pd.DataFrame(spearman_results)
sp_df['q_BH'] = multipletests(sp_df['p'], method='fdr_bh')[1]
print("\\n=== Pathway completeness vs predicted fitness (Spearman) ===")
print(sp_df.sort_values('rho', ascending=False).to_string(index=False))
sp_df.to_csv(DATA_DIR / 'nb03_pathway_fitness_spearman.csv', index=False)
'''
print(scaffold)

## Section 5 — Spark Scaffold: Contamination Gradient Analysis

Test whether MAI prevalence across SSO samples/sites correlates with metal contamination level.
This requires joining `enigma_all_genomes` (genome_id → has_mai) with site-level metadata
(sample → site → contamination level) from the ENIGMA SSO tables.

In [ ]:
# Preview contamination gradient context from enigma_contamination_functional_potential
model_results = pd.read_csv(CONTAM_DIR / 'data' / 'model_results.tsv', sep='\t')
site_scores   = pd.read_csv(CONTAM_DIR / 'data' / 'site_functional_scores.tsv', sep='\t')

# Key result: defense signal survives covariate adjustment (p=0.0004) but not global FDR
defense_row = model_results[
    (model_results['outcome'] == 'site_defense_score') &
    (model_results['mapping_mode'] == 'relaxed_all_clades')
].iloc[0]

print('Prior result from enigma_contamination_functional_potential:')
print(f'  Site-level defense ~ contamination (Spearman rho={defense_row.spearman_rho:.3f}, p={defense_row.spearman_p:.3f})')
print(f'  Covariate-adjusted beta={defense_row.adj_cov_beta_contamination:.6f}, p={defense_row.adj_cov_p_contamination:.4f}')
print(f'  This p=0.0004 signal does NOT survive global FDR across all defense/metabolism/mobilome outcomes.')
print()
print('New focused test: MAI prevalence specifically (not all defense genes).')
print('This is a single pre-specified test, not a sweep — FDR correction less severe.')
print()
print('Spark scaffold for MAI-contamination correlation:')

scaffold_contam = '''
# ── S5a: Get ENIGMA SSO sample-genome associations ────────────────────────────
# (Requires Spark + ENIGMA SSO tables)
sso_genome_sql = \'\'\'\n    SELECT
        s.sdt_sample_name,
        g.genome_id,
        g.has_mai
    FROM enigma_sso_sample_genome_bridge s   -- table name TBD; check enigma_sso_asv_ecology NB01
    JOIN enigma_all_genomes g ON s.genome_id = g.genome_id
\'\'\'\nsso_df = spark.sql(sso_genome_sql).toPandas()

# ── S5b: Compute per-site MAI prevalence ─────────────────────────────────────
site_mai = sso_df.groupby('sdt_sample_name').agg(
    n_genomes  = ('genome_id', 'count'),
    n_mai      = ('has_mai', 'sum')
).reset_index()
site_mai['mai_fraction'] = site_mai['n_mai'] / site_mai['n_genomes']

# ── S5c: Join with contamination index ────────────────────────────────────────
contam_meta = pd.read_csv(CONTAM_DIR / 'data' / 'geochemistry_sample_matrix.tsv', sep='\\t')
site_joined = site_mai.merge(contam_meta, on='sdt_sample_name', how='inner')

# ── S5d: Spearman correlation: MAI fraction vs contamination ─────────────────
rho, p = spearmanr(site_joined['mai_fraction'], site_joined['contamination_index'])
print(f"MAI fraction vs contamination: Spearman rho={rho:.3f}, p={p:.4f}")
print(f"n = {len(site_joined)} samples")
# Expected: positive rho if MAI is adaptive under contamination
'''
print(scaffold_contam)

## Section 6 — Experimental Priorities

Given the data limitations, prioritize experiments that can resolve the MAI hypothesis
without depending on Spark results or Jen's isolate list.

In [ ]:
# Top candidates: complete pathway + Streptomyces (genetic tools available)
top_cands = shortlist[shortlist['pathway_completeness'] == 1.0].copy()
top_cands = top_cands[top_cands.get('genus', top_cands.get('organism_name', pd.Series()).str.split().str[0]) == 'Streptomyces']

if 'genus' not in top_cands.columns:
    top_cands['genus'] = top_cands['organism_name'].str.split().str[0]

strep_cands = shortlist[shortlist['organism_name'].str.startswith('Streptomyces')].head(5)

print('=== TOP EXPERIMENTAL PRIORITIES ===')
print()
print('Tier 1 — Streptomyces strains (genetic tools, full pathway):')
print(strep_cands[['rank', 'organism_name', 'pathway_completeness', 'score_total']].to_string(index=False))
print()
print('Recommended experiments (priority order):')
print('  1. MIC assay: Cu²⁺, Cd²⁺, Zn²⁺, Hg²⁺ (4 metals predicted most affected by mycothiol)')
print('  2. Growth curve fitting: capture µmax and lag time, not just endpoint OD')
print('     → lag time likely more sensitive to MAI function than growth rate')
print('  3. MAI knockout: construct Δmai in Streptomyces mirabilis YR139 (top candidate)')
print('     → compare MIC WT vs Δmai vs complemented')
print('  4. MSH-metal adduct assay: in vitro test whether MAI processes mycothiol-Cu²⁺ substrate')
print()
print('Key prediction from the mycothiol hypothesis:')
print('  MAI effect should be largest for Cu²⁺ and Hg²⁺ (known to form stable thiol adducts)')
print('  Effects for Zn²⁺ and Cd²⁺ may be smaller (weaker thiol affinity)')
print()
print('Context from genotype_to_phenotype_enigma (null result):')
print('  Metal growth AUC ≈ 0.5 from KO content → endpoint OD600 may not detect the effect')
print('  Recommend: use growth curve µmax + lag time, not binary endpoint, as phenotype readout')

In [ ]:
print('=== NB03 SUMMARY ===')
print()
print('DATA LANDSCAPE:')
print(f'  ENIGMA genomes with MAI status: {len(all_genomes):,} (has_mai: {all_genomes["has_mai"].sum():,} positive)')
print(f'  Genus overlap with metal growth data: {len(overlap_gen)} genera (insufficient for local analysis)')
print()
print('genotype_to_phenotype_enigma NULL RESULT:')
print('  Metal growth not predictable from KO content (AUC ≈ 0.5).')
print('  Interpret cautiously: endpoint OD600 likely insensitive to MAI-mediated lag effects.')
print()
print('PENDING (requires Spark):')
print('  Section 4: Apply enigma_stress_phenotype_ml models to all Genome Depot proteins')
print('  Section 5: MAI prevalence vs contamination gradient across SSO sites')
print()
print('EXPERIMENTAL RECOMMENDATION:')
print('  Prioritize growth curve (µmax + lag) assay for top-5 Streptomyces candidates.')
print('  Test Cu²⁺ and Hg²⁺ first (strongest predicted thiol-metal interaction).')
print('  Build Δmai knockout in Streptomyces mirabilis YR139 for definitive test.')